# TOPOTEX Orientation-Aligned Dataset Inspector

主线数据：**TexVerse-OA 1K**（`topotex_data_OA/texverse`）——canonical
mesh（VLM 四视图选正面 + 90° Y 轴旋转）+ 6 张随机相机渲染 + native /
xatlas / connected-partial 三查询。`DATASET_MODE="objaverse_oa"` 可切回
OA-80 prototype（研究记录）。相机参数只是记录用元数据，绝不输入模型;
**final_test 对象在训练结束前不展示**。

In [ ]:
from pathlib import Path

DATASET_MODE = "texverse_oa"   # "texverse_oa" | "objaverse_oa"
SAMPLE_ID = None               # 指定对象 id；None = 随机
RANDOM_SEED = None             # None = 每次执行随机；设整数可复现
N_GALLERY = 16
MODE = "quick"                 # "quick" | "full"（full: 一致性扫全部对象）

OA_BASE = Path("/root/youjiaZhang/topotex_data_OA")
TX = DATASET_MODE == "texverse_oa"
ROOT = OA_BASE / "texverse" if TX else OA_BASE
HUMAN = ROOT / "objects" if TX else ROOT / "oa100"
DS = ROOT / "dataset"

In [ ]:
import hashlib, json, os
import numpy as np
import matplotlib.pyplot as plt
import trimesh
from PIL import Image
from safetensors.numpy import load_file

man = [json.loads(l) for l in open(DS / "manifest.jsonl")] if (DS / "manifest.jsonl").exists() else \
      [{"sample_id": p.name} for p in sorted((DS / "samples").iterdir()) if (p / "meta.json").exists()]
uids_all = [m["sample_id"] for m in man]
split_p = ROOT / "object_split.json"
if split_p.exists():
    SPLIT = json.loads(split_p.read_text())
    VISIBLE = sorted(set(SPLIT["train"]) | set(SPLIT["validation"]))
    split_of = {u: s for s in ("train", "validation", "final_test") for u in SPLIT[s]}
else:
    SPLIT, VISIBLE, split_of = None, uids_all, {}
rng = np.random.default_rng(RANDOM_SEED)
print(f"mode={DATASET_MODE} | objects={len(uids_all)} | visible(train+val)={len(VISIBLE)} | MODE={MODE}")

## Section 0 — Dataset Overview

In [ ]:
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
metas = {u: json.loads((DS / "samples" / u / "meta.json").read_text()) for u in uids_all}
faces = np.array([metas[u]["num_faces"] for u in uids_all])
verts = np.array([metas[u]["num_vertices"] for u in uids_all])
if TX:
    sel = json.loads((ROOT / "selection.json").read_text())
    fv = json.loads((ROOT / "front_votes_meta.json").read_text())
    print(f"selection             {sel['n_selected']} of pool ({sel['rule']})")
    print(f"selection SHA         {open(ROOT / 'selection.sha256').read().strip()}")
    print(f"front votes           {fv['n_votes']} (low-confidence {fv['n_low_confidence']}) hist {fv['front_hist']}")
    print(f"VLM                   {fv['vlm']} | prompt sha {fv['prompt_sha256'][:16]}…")
    if SPLIT:
        print(f"split                 {SPLIT['n_train']}/{SPLIT['n_validation']}/{SPLIT['n_final_test']} seed {SPLIT['seed']} | sha {open(ROOT / 'object_split.sha256').read().strip()[:16]}…")
if (DS / "manifest.jsonl").exists():
    print(f"manifest SHA          {sha(DS / 'manifest.jsonl')}")
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].hist(faces, bins=30); axes[0].set_title(f"face count (med {int(np.median(faces))} max {faces.max()})", fontsize=9)
axes[1].hist(verts, bins=30); axes[1].set_title(f"vertex count (med {int(np.median(verts))})", fontsize=9)
plt.tight_layout(); plt.show()

## Section 1 — Random Object（含 orientation 选择证据）

In [ ]:
UID = SAMPLE_ID or VISIBLE[int(rng.integers(len(VISIBLE)))]
d = DS / "samples" / UID
h = HUMAN / UID
m = metas[UID]
tr = json.loads((h / "mesh/transform.json").read_text())
if TX and (ROOT / "previews" / f"{UID}.png").exists():
    prev = np.asarray(Image.open(ROOT / "previews" / f"{UID}.png"))
    k = tr["chosen_front_index"]
    fig, ax = plt.subplots(figsize=(16, 4))
    ax.imshow(prev)
    ax.add_patch(plt.Rectangle((k * 256, 0), 256, 256, fill=False, ec="lime", lw=3))
    ax.set_title(f"orientation preview (original pose, az 0/90/180/270) — VLM chose view {k} as semantic front", fontsize=10)
    ax.axis("off"); plt.show()
sc = trimesh.load(str(h / "mesh/canonical.glb"), force="mesh", process=False)
V = np.asarray(sc.vertices); Vn = (V - V.mean(0)) / max(np.abs(V - V.mean(0)).max(), 1e-9)
idx = np.random.default_rng(0).choice(len(Vn), min(5000, len(Vn)), replace=False)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (t, a, b) in zip(axes, [("XY (front = +Z toward viewer)", 0, 1), ("ZY", 2, 1), ("XZ top", 0, 2)]):
    ax.scatter(Vn[idx, a], Vn[idx, b], s=0.5, c="k")
    ax.arrow(0, 0, .45, 0, color="r", width=.008); ax.arrow(0, 0, 0, .45, color="g", width=.008)
    ax.set_title(f"canonical mesh {t}", fontsize=9); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
plt.show()
print(f"object            {UID} | split {split_of.get(UID, 'n/a')} | category n/a (TexVerse ships no labels)" if TX
      else f"object            {UID}")
print(f"faces/vertices    {m['num_faces']} / {m['num_vertices']} | texture stored {m['texture_resolution']}")
print(f"orientation_aligned {tr['orientation_aligned']} | chosen front {tr.get('chosen_front_index', 'n/a')}")
print(f"R = {np.array(tr['R']).round(4).tolist()}")
print(f"t = {tr['t']} | s = {tr['s']} | source glb sha {tr.get('source_glb_sha256', 'n/a')[:16]}…")

## Section 2 — Stochastic Views + 3D 相机位姿（交互式）

In [ ]:
cam_f = h / "images/cameras.json" if TX else h / "images/view_meta.json"
vm = json.loads(cam_f.read_text())
fig, axes = plt.subplots(1, 6, figsize=(21, 3.6))
for k in range(6):
    axes[k].imshow(Image.open(h / f"images/view_{k:03d}.png"))
    axes[k].set_title(f"view_{k:03d}  az {vm[k]['azimuth']:.1f} el {vm[k]['elevation']:.1f}", fontsize=8)
    axes[k].axis("off")
plt.show()
import sys
sys.path.insert(0, "/root/youjiaZhang/TopoTex")
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook"
from topotex.data.mesh import camera_matrices
ms = load_file(str(d / "mesh.safetensors"))
Vm, Fm = ms["vertices"], ms["faces"].astype(np.int64)
bmin, bmax = Vm.min(0).astype(np.float64), Vm.max(0).astype(np.float64)
ctr3 = (bmin + bmax) / 2
eyes = np.stack([np.linalg.inv(camera_matrices(c_["azimuth"], c_["elevation"], bmin, bmax)["view"])[:3, 3] for c_ in vm])
pal = ["#e6194b", "#3cb44b", "#4363d8", "#f58231", "#911eb4", "#42d4f4"]
traces = [go.Mesh3d(x=Vm[:, 0], y=Vm[:, 1], z=Vm[:, 2], i=Fm[:, 0], j=Fm[:, 1], k=Fm[:, 2],
                    color="lightsteelblue", flatshading=True, lighting=dict(ambient=0.45, diffuse=0.8))]
for k, e in enumerate(eyes):
    traces += [go.Scatter3d(x=[e[0]], y=[e[1]], z=[e[2]], mode="markers+text", marker=dict(size=6, color=pal[k]),
                            text=[f"v{k}"], textposition="top center", showlegend=False),
               go.Scatter3d(x=[e[0], ctr3[0]], y=[e[1], ctr3[1]], z=[e[2], ctr3[2]], mode="lines",
                            line=dict(color=pal[k], width=3), showlegend=False)]
axl = float(np.linalg.norm(bmax - bmin)) * 0.4
for vec, col, nm in ((np.array([1, 0, 0]), "red", "+X"), (np.array([0, 1, 0]), "green", "+Y up"), (np.array([0, 0, 1]), "blue", "+Z front")):
    q = ctr3 + vec * axl
    traces.append(go.Scatter3d(x=[ctr3[0], q[0]], y=[ctr3[1], q[1]], z=[ctr3[2], q[2]], mode="lines+text",
                               line=dict(color=col, width=5), text=["", nm], showlegend=False))
fig3d = go.Figure(traces)
fig3d.update_layout(scene=dict(aspectmode="data"), height=600, width=800, margin=dict(l=0, r=0, t=28, b=0),
                    title=f"{UID[:12]} — canonical mesh + camera poses")
fig3d.show()

## Section 3 — Texture and UV Queries

**partial is a surface subset query, not an unwrap family**。

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 9.2))
for r_i, (q, sem) in enumerate([("uv_000", "native full layout"), ("uv_001", "xatlas full layout"),
                                ("uv_002", "connected partial query")]):
    qa = load_file(str(d / f"uv_queries/{q}/uv_address.safetensors"))
    gt = plt.imread(d / f"uv_queries/{q}/gt_texture.png")
    fid = qa["face_id"]; bar = qa["barycentric"].astype(np.float32); val = qa["valid_mask"].astype(bool)
    uvv = qa["uv_vertices"]
    axes[r_i, 0].scatter(uvv[:, 0], 1 - uvv[:, 1], s=0.12, c="k"); axes[r_i, 0].set_aspect("equal")
    axes[r_i, 0].set_title(f"{sem}\n({len(uvv)} uv verts)", fontsize=8)
    axes[r_i, 1].imshow(gt); axes[r_i, 1].set_title("GT texture", fontsize=8)
    axes[r_i, 2].imshow(np.where(val, fid, -1), cmap="nipy_spectral"); axes[r_i, 2].set_title(f"face_id (max {fid.max()})", fontsize=8)
    axes[r_i, 3].imshow(np.clip(bar.transpose(1, 2, 0), 0, 1) * val[..., None]); axes[r_i, 3].set_title("barycentric RGB", fontsize=8)
    axes[r_i, 4].imshow(val, cmap="gray"); axes[r_i, 4].set_title(f"valid mask ({val.mean() * 100:.1f}%)", fontsize=8)
    for ax in axes[r_i]: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## Section 4 — Consistency Checks（16 随机 train/validation 对象）

In [ ]:
def check_object(uid):
    dd = DS / "samples" / uid; hh = HUMAN / uid
    out = {}
    try:
        ms_ = load_file(str(dd / "mesh.safetensors")); F = ms_["faces"]
        out["mesh readable"] = bool(ms_["vertices"].ndim == 2 and len(F) > 0)
        out["no NaN/Inf"] = bool(np.isfinite(ms_["vertices"]).all()) and bool(np.isfinite(ms_["uv_vertices"]).all())
        st = load_file(str(dd / "mv.safetensors"))
        out["image completeness"] = bool(st["images"].shape == (6, 3, 256, 256) and st["images"].dtype == np.uint8
                                         and all((hh / f"images/view_{k:03d}.png").exists() for k in range(6)))
        out["texture readable"] = bool(plt.imread(dd / "gt_texture.png").shape[:2] == (256, 256))
        tr_ = json.loads((hh / "mesh/transform.json").read_text())
        out["orientation metadata"] = bool(tr_.get("orientation_aligned") is True and np.array(tr_["R"]).shape == (3, 3)
                                           and ("chosen_front_index" in tr_ if TX else True))
        out["render non-empty"] = bool(all(np.asarray(Image.open(hh / f"images/view_{k:03d}.png")).std() > 2 for k in range(6)))
        okf = okb = okm = True
        for q in ("uv_000", "uv_001", "uv_002"):
            qa = load_file(str(dd / f"uv_queries/{q}/uv_address.safetensors"))
            v = qa["valid_mask"].astype(bool); fidv = qa["face_id"][v]
            bs = qa["barycentric"].astype(np.float32).transpose(1, 2, 0)[v].sum(-1)
            okf &= bool(v.any()) and bool(fidv.min() >= 0) and bool(fidv.max() < len(F))
            okb &= bool(np.abs(bs - 1).max() < 2e-2) and bool(np.isfinite(qa["barycentric"].astype(np.float32)).all())
            okm &= bool((qa["face_id"][~v] == -1).all())
        out["UV face_id range"] = bool(okf); out["barycentric sum"] = bool(okb); out["mask consistent"] = bool(okm)
        tex_h = hh / ("texture/native_texture.png" if TX else "texture/gt_texture.png")
        out["same object identity"] = bool(os.stat(dd / "gt_texture.png").st_ino == os.stat(tex_h).st_ino
                                           and os.stat(dd / "meta.json").st_ino == os.stat(hh / "metadata.json").st_ino)
    except Exception as e:
        out["exception"] = f"FAIL {type(e).__name__}: {e}"
    return out

GALLERY = sorted(set([UID] + [VISIBLE[i] for i in rng.choice(len(VISIBLE), N_GALLERY - 1, replace=False)]))[:N_GALLERY]
scan = uids_all if MODE == "full" else GALLERY
n_bad = 0
for u in scan:
    r = check_object(u)
    bad = not all(v is True for v in r.values())
    n_bad += bad
    if bad: print("!!", u, r)
print(f"consistency: {len(scan) - n_bad}/{len(scan)} objects pass "
      f"({'ALL' if MODE == 'full' else f'{len(scan)} random train/validation'}); checks/object = {len(check_object(UID))}")

## Section 5 — Gallery（16 随机 train/validation 对象）

In [ ]:
for uid in GALLERY:
    dd = DS / "samples" / uid; hh = HUMAN / uid
    mm = metas[uid]; tr_ = json.loads((hh / "mesh/transform.json").read_text())
    qa = load_file(str(dd / "uv_queries/uv_000/uv_address.safetensors"))
    fig, axes = plt.subplots(1, 9, figsize=(27, 3.0))
    if TX and (ROOT / "previews" / f"{uid}.png").exists():
        prev = np.asarray(Image.open(ROOT / "previews" / f"{uid}.png"))
        k = tr_["chosen_front_index"]
        axes[0].imshow(prev[:, k * 256:(k + 1) * 256]); axes[0].set_title(f"chosen front (v{k})", fontsize=8)
    for k2 in range(6):
        axes[1 + k2].imshow(Image.open(hh / f"images/view_{k2:03d}.png")); axes[1 + k2].set_title(f"v{k2}", fontsize=8)
    axes[7].imshow(plt.imread(dd / "gt_texture.png")); axes[7].set_title("native texture", fontsize=8)
    axes[8].imshow(qa["valid_mask"].astype(bool), cmap="gray"); axes[8].set_title("native UV mask", fontsize=8)
    for ax in axes: ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f"{uid} | split {split_of.get(uid, 'n/a')} | faces {mm['num_faces']} | aligned {tr_['orientation_aligned']}", fontsize=9, y=1.03)
    plt.tight_layout(); plt.show()

import os as _os
outd = Path(_os.environ.get("TOPOTEX_RUN_ROOT", str(OA_BASE / "runs"))) / "dataset_inspector_oa"
outd.mkdir(parents=True, exist_ok=True)
stats = {"mode": DATASET_MODE, "n_objects": len(uids_all),
         "face_count": {"min": int(faces.min()), "median": int(np.median(faces)), "max": int(faces.max())},
         "split": {k: SPLIT[f"n_{k}" if k != "train" else "n_train"] for k in ("train", "validation", "final_test")} if SPLIT else None,
         "consistency": {"scanned": len(scan), "failed": n_bad}}
json.dump(stats, open(outd / f"{DATASET_MODE}_statistics.json", "w"), indent=1)
print("stats ->", outd / f"{DATASET_MODE}_statistics.json")